# Schwarzschild (1906) — On the Equilibrium of the Solar Atmosphere
# 태양 대기의 평형에 관하여 — 구현

This notebook implements and visualizes the key concepts from Schwarzschild's 1906 paper:
이 노트북은 Schwarzschild의 1906년 논문의 핵심 개념을 구현하고 시각화합니다:

1. **Part 1**: Three types of atmospheric equilibrium / 세 가지 대기 평형
2. **Part 2**: Radiative transfer equations from scratch / 복사 전달 방정식 직접 구현
3. **Part 3**: Temperature-depth profiles / 온도-깊이 프로파일
4. **Part 4**: Limb darkening prediction / 주변감광 예측
5. **Part 5**: Comparison with Schwarzschild's observational data / 관측 데이터 비교
6. **Part 6**: Stability analysis / 안정성 분석
7. **Part 7**: Solar disk visualization / 태양 원반 시각화
8. **Part 8**: Connection to modern astrophysics / 현대 천체물리학과의 연결

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colormaps
from scipy.integrate import quad

plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

# Physical constants in Schwarzschild's unit system.
T_EFF = 6000.0       # Effective temperature (K).
TAU_BOUNDARY = T_EFF / np.sqrt(2)  # Boundary temperature (~5050 K).
G_SUN = 27.7         # Solar surface gravity (Earth units).
R_GAS = 0.106        # Gas constant in Schwarzschild's units.
M_AIR = 28.9         # Molecular weight of air.
K_ABS = 0.6          # Absorption coefficient for air.

print(f"Effective temperature T = {T_EFF} K")
print(f"Boundary temperature τ = T/√2 = {TAU_BOUNDARY:.0f} K")
print(f"Solar gravity g = {G_SUN} × Earth")

## Part 1: Three Types of Atmospheric Equilibrium / 세 가지 대기 평형

Schwarzschild는 태양 대기의 세 가지 평형 상태를 비교합니다:
- **등온 평형(Isothermal)**: $t = \text{const}$ — 온도 일정 / constant temperature
- **단열 평형(Adiabatic)**: $t \propto h$ — 대류에 의한 선형 온도 경사 / linear gradient from convection
- **복사 평형(Radiative)**: $t^4 \propto (1 + m)$ — 복사 흡수/방출 균형 / radiation absorption/emission balance

$$\text{등온: } p = p_0 e^{\frac{Mg}{Rt}h}, \qquad \text{단열: } t - t_0 = \frac{k-1}{k}\frac{Mg}{R}(h-h_0), \qquad \text{복사: } t^4 = \tau^4(1 + \frac{k}{g}p)$$

In [ ]:
def isothermal_profile(h, t0=TAU_BOUNDARY, M=M_AIR, g=G_SUN, R=R_GAS):
    """Compute isothermal atmospheric profile.

    Args:
        h: Depth array (positive inward, units of 8 km).
        t0: Constant temperature (K).
        M, g, R: Molecular weight, gravity, gas constant.

    Returns:
        Tuple of (temperature, pressure, density) arrays.
    """
    exponent = M * g / (R * t0) * h
    p = np.exp(exponent)  # Normalized to p=1 at h=0.
    rho = p * M / (R * t0)
    t = np.full_like(h, t0)
    return t, p, rho


def adiabatic_profile(h, t0=TAU_BOUNDARY, k=4/3, M=M_AIR, g=G_SUN, R=R_GAS):
    """Compute adiabatic atmospheric profile.

    Args:
        h: Depth array.
        t0: Temperature at h=0.
        k: Ratio of specific heats (cp/cv).

    Returns:
        Tuple of (temperature, pressure, density) arrays.
    """
    grad = (k - 1) / k * M * g / R  # Temperature gradient (K per unit h).
    t = t0 + grad * h
    t = np.maximum(t, 0)  # Temperature cannot be negative.
    p = (t / t0) ** (k / (k - 1))
    rho = p * M / (R * t)
    rho[t <= 0] = 0
    return t, p, rho


def radiative_profile(h, T=T_EFF, k_abs=K_ABS, k_gas=4/3,
                       M=M_AIR, g=G_SUN, R=R_GAS):
    """Compute radiative equilibrium atmospheric profile.

    Uses Schwarzschild's eq. (14): t^4 = T^4/2 * [1 + k_abs/g * p].
    Solved iteratively since p depends on the integral of rho*g*dh.

    Args:
        h: Depth array (must be evenly spaced).
        T: Effective temperature.
        k_abs: Absorption coefficient.

    Returns:
        Tuple of (temperature, optical_depth, density) arrays.
    """
    tau = T / np.sqrt(2)
    # For grey atmosphere: m = k_abs * p / g.
    # t^4 = tau^4 * (1 + m), and m relates to p.
    # Use pressure as the independent variable.
    n_points = len(h)
    t = np.zeros(n_points)
    m = np.zeros(n_points)
    rho = np.zeros(n_points)

    dh = h[1] - h[0] if len(h) > 1 else 1.0
    p_current = 0.0001  # Small initial pressure at the top.

    for i in range(n_points):
        m_val = k_abs * p_current / g
        t4 = tau**4 * (1 + m_val)
        t_val = t4 ** 0.25
        rho_val = p_current * M / (R * t_val)

        t[i] = t_val
        m[i] = m_val
        rho[i] = rho_val

        # Hydrostatic step.
        p_current += rho_val * g * dh

    return t, m, rho


# Compute profiles.
h = np.linspace(-5, 8, 500)  # Depth in units of 8 km.

t_iso, _, _ = isothermal_profile(h)
t_adi, _, _ = adiabatic_profile(h, k=4/3)
t_rad, m_rad, _ = radiative_profile(h)

# Plot temperature profiles.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Left: Temperature vs depth.
ax1.plot(h * 8, t_iso, 'b--', linewidth=2, label='Isothermal / 등온')
ax1.plot(h * 8, t_adi, 'r-', linewidth=2, label='Adiabatic / 단열 (k=4/3)')
ax1.plot(h * 8, t_rad, 'g-', linewidth=2.5, label='Radiative / 복사')
ax1.axhline(TAU_BOUNDARY, color='blue', alpha=0.3, linestyle=':')
ax1.text(10, TAU_BOUNDARY + 200, f'τ = {TAU_BOUNDARY:.0f} K',
         color='blue', fontsize=10)
ax1.axhline(T_EFF, color='orange', alpha=0.3, linestyle=':')
ax1.text(10, T_EFF + 200, f'T_eff = {T_EFF} K', color='orange', fontsize=10)
ax1.set_xlabel("Depth / 깊이 (km)", fontsize=13)
ax1.set_ylabel("Temperature / 온도 (K)", fontsize=13)
ax1.set_title("Temperature vs Depth: Three Equilibria\n"
              "온도 vs 깊이: 세 가지 평형", fontsize=14)
ax1.set_ylim(0, 25000)
ax1.legend(fontsize=11)
ax1.grid(alpha=0.3)

# Right: Temperature vs optical depth (radiative only).
ax2.plot(m_rad, t_rad, 'g-', linewidth=2.5, label='Radiative / 복사')
ax2.axhline(TAU_BOUNDARY, color='blue', alpha=0.3, linestyle=':')
ax2.text(0.5, TAU_BOUNDARY - 300, f'τ = T/√2 = {TAU_BOUNDARY:.0f} K',
         color='blue', fontsize=10)
ax2.axvline(1, color='gray', alpha=0.3, linestyle='--')
ax2.text(1.1, 5500, 'm = 1', color='gray', fontsize=10)
ax2.set_xlabel("Optical Depth m / 광학적 깊이", fontsize=13)
ax2.set_ylabel("Temperature / 온도 (K)", fontsize=13)
ax2.set_title("Temperature vs Optical Depth\n"
              "온도 vs 광학적 깊이 (복사 평형)", fontsize=14)
ax2.set_xlim(0, 5)
ax2.set_ylim(4000, 12000)
ax2.legend(fontsize=11)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("핵심 관찰:")
print(f"  등온: 온도 일정 ({TAU_BOUNDARY:.0f} K)")
print(f"  단열: 온도가 깊이에 따라 선형 증가")
print(f"  복사: t⁴가 광학적 깊이 m에 따라 선형 증가 → t는 m^(1/4)로 완만히 증가")

## Part 2: Radiative Transfer Equations from Scratch / 복사 전달 방정식 직접 구현

Schwarzschild의 복사 전달 방정식을 수치적으로 풀어봅니다. 두 방향의 복사를 광학적 깊이 $m$의 함수로 구합니다:

Numerically solve Schwarzschild's radiative transfer equations for two-stream radiation as a function of optical depth $m$:

$$\frac{dB}{dm} = E - B, \qquad \frac{dA}{dm} = A - E$$

해석해 / Analytical solution: $E = \frac{A_0}{2}(1+m)$, $A = \frac{A_0}{2}(2+m)$, $B = \frac{A_0}{2}m$

In [ ]:
def solve_radiative_transfer_numerical(m_max=5.0, n_steps=1000, A0=1.0):
    """Solve Schwarzschild's two-stream radiative transfer numerically.

    Uses Euler method to solve:
        dB/dm = E - B
        dA/dm = A - E
    with radiative equilibrium condition A + B = 2E.

    Args:
        m_max: Maximum optical depth.
        n_steps: Number of integration steps.
        A0: Outgoing radiation at the surface.

    Returns:
        Tuple of (m, A, B, E) arrays.
    """
    dm = m_max / n_steps
    m = np.linspace(0, m_max, n_steps + 1)
    A = np.zeros(n_steps + 1)
    B = np.zeros(n_steps + 1)
    E = np.zeros(n_steps + 1)

    # Boundary conditions at m=0: B(0) = 0, A(0) = A0.
    B[0] = 0.0
    A[0] = A0
    E[0] = (A[0] + B[0]) / 2  # Radiative equilibrium: A + B = 2E.

    for i in range(n_steps):
        dB = (E[i] - B[i]) * dm
        dA = (A[i] - E[i]) * dm

        B[i + 1] = B[i] + dB
        A[i + 1] = A[i] + dA
        E[i + 1] = (A[i + 1] + B[i + 1]) / 2

    return m, A, B, E


def analytical_solution(m, A0=1.0):
    """Schwarzschild's analytical solution (eq. 11).

    Args:
        m: Optical depth array.
        A0: Outgoing radiation at the surface.

    Returns:
        Tuple of (A, B, E) arrays.
    """
    E = A0 / 2 * (1 + m)
    A = A0 / 2 * (2 + m)
    B = A0 / 2 * m
    return A, B, E


# Solve both ways.
m_num, A_num, B_num, E_num = solve_radiative_transfer_numerical(A0=1.0)
A_ana, B_ana, E_ana = analytical_solution(m_num, A0=1.0)

# Plot.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Left: Numerical solution.
ax1.plot(m_num, A_num, 'r-', linewidth=2, label='A (outward / 바깥)')
ax1.plot(m_num, B_num, 'b-', linewidth=2, label='B (inward / 안쪽)')
ax1.plot(m_num, E_num, 'k-', linewidth=2.5, label='E = ct⁴ (blackbody / 흑체)')
ax1.fill_between(m_num, B_num, A_num, alpha=0.1, color='purple')
ax1.set_xlabel("Optical Depth m / 광학적 깊이", fontsize=13)
ax1.set_ylabel("Radiation Energy / 복사 에너지", fontsize=13)
ax1.set_title("Numerical Solution of RT Equations\n"
              "복사 전달 방정식의 수치 해", fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(alpha=0.3)
ax1.annotate('A + B = 2E\n(radiative eq.)',
             xy=(3, E_num[600]), fontsize=10,
             bbox=dict(boxstyle='round', fc='lightyellow'))

# Right: Verify against analytical.
ax2.plot(m_num, np.abs(A_num - A_ana), 'r-', label='|A_num - A_ana|')
ax2.plot(m_num, np.abs(B_num - B_ana), 'b-', label='|B_num - B_ana|')
ax2.plot(m_num, np.abs(E_num - E_ana), 'k-', label='|E_num - E_ana|')
ax2.set_xlabel("Optical Depth m / 광학적 깊이", fontsize=13)
ax2.set_ylabel("Absolute Error / 절대 오차", fontsize=13)
ax2.set_title("Numerical vs Analytical Error\n"
              "수치 해 vs 해석 해 오차", fontsize=14)
ax2.legend(fontsize=11)
ax2.grid(alpha=0.3)
ax2.set_yscale('log')

plt.tight_layout()
plt.show()

print(f"최대 오차: A={np.max(np.abs(A_num-A_ana)):.2e}, "
      f"B={np.max(np.abs(B_num-B_ana)):.2e}")
print(f"\nm=0 (표면): A={A_num[0]:.3f}, B={B_num[0]:.3f}, E={E_num[0]:.3f}")
print(f"  → B=0: 바깥에서 안으로 들어오는 복사 없음 (경계 조건)")
print(f"  → E=A₀/2: 표면 온도 t⁴ = T⁴/2, 즉 τ = T/√2")

## Part 3–5: Limb Darkening — The Key Prediction / 주변감광 — 핵심 예측

Schwarzschild의 가장 중요한 결과: 두 모델의 주변감광 법칙과 관측 데이터 비교.

Schwarzschild's most important result: limb darkening laws from both models compared with observations.

**복사 평형 / Radiative eq.**: $F(i) = \dfrac{1 + 2\cos i}{3}$ (eq. 28)

**단열 평형 / Adiabatic eq.** ($k=4/3$): $F(i) = \cos i$ (eq. 29)

태양 원반에서의 위치 $r/R$과 각도 $i$의 관계: $\sin i = r/R$ (eq. 23)

In [ ]:
def limb_darkening_radiative(r_over_R):
    """Schwarzschild's radiative equilibrium limb darkening (eq. 28).

    Args:
        r_over_R: Fractional distance from disk center (0=center, 1=limb).

    Returns:
        Normalized brightness F(i), center = 1.
    """
    cos_i = np.sqrt(1 - r_over_R**2)
    return (1 + 2 * cos_i) / 3


def limb_darkening_adiabatic(r_over_R, k=4/3):
    """Schwarzschild's adiabatic equilibrium limb darkening (eq. 29).

    Args:
        r_over_R: Fractional distance from disk center.
        k: Ratio of specific heats.

    Returns:
        Normalized brightness F(i), center = 1.
    """
    cos_i = np.sqrt(1 - np.minimum(r_over_R, 0.9999)**2)
    return cos_i ** (4 * (k - 1) / k)


# Observational data from Schwarzschild's paper (p. 52, G. Müller).
obs_r_R = np.array([0.0, 0.2, 0.4, 0.6, 0.7, 0.8, 0.9, 0.96, 0.98, 1.0])
obs_brightness = np.array([1.00, 0.99, 0.97, 0.92, 0.87, 0.81, 0.70,
                            0.59, 0.49, 0.40])

# Compute theoretical predictions.
r = np.linspace(0, 0.999, 500)
F_rad = limb_darkening_radiative(r)
F_adi = limb_darkening_adiabatic(r)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Left: Limb darkening curves.
ax1.plot(r, F_rad, 'g-', linewidth=2.5,
         label='Radiative eq. (28) / 복사 평형')
ax1.plot(r, F_adi, 'r--', linewidth=2.5,
         label='Adiabatic eq. (29) / 단열 평형')
ax1.plot(obs_r_R, obs_brightness, 'ko', markersize=8, zorder=5,
         label='Observations (Müller) / 관측')
ax1.set_xlabel("r/R (fractional distance from center / 중심으로부터 거리)",
               fontsize=12)
ax1.set_ylabel("F(i) (normalized brightness / 정규화 밝기)", fontsize=12)
ax1.set_title("Limb Darkening: Theory vs Observation\n"
              "주변감광: 이론 vs 관측 (Schwarzschild 1906, p.52)",
              fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(alpha=0.3)
ax1.set_ylim(0, 1.1)

# Highlight key differences.
ax1.annotate('Adiabatic → 0 at limb!\n단열: 가장자리에서 0!',
             xy=(1.0, 0.0), xytext=(0.75, 0.15),
             fontsize=10, color='red',
             arrowprops=dict(arrowstyle='->', color='red'))
ax1.annotate(f'Radiative → 1/3 at limb\n복사: 가장자리에서 1/3',
             xy=(0.99, 0.34), xytext=(0.6, 0.42),
             fontsize=10, color='green',
             arrowprops=dict(arrowstyle='->', color='green'))

# Right: Residuals (observation - theory).
resid_rad = obs_brightness - limb_darkening_radiative(obs_r_R)
resid_adi = obs_brightness - limb_darkening_adiabatic(obs_r_R)

ax2.bar(obs_r_R - 0.015, resid_rad, width=0.025, color='green',
        alpha=0.7, label='Obs − Radiative / 관측 − 복사')
ax2.bar(obs_r_R + 0.015, resid_adi, width=0.025, color='red',
        alpha=0.7, label='Obs − Adiabatic / 관측 − 단열')
ax2.axhline(0, color='black', linewidth=0.5)
ax2.set_xlabel("r/R", fontsize=12)
ax2.set_ylabel("Residual (Obs − Theory) / 잔차", fontsize=12)
ax2.set_title("Residuals: Which Model Fits Better?\n"
              "잔차: 어느 모델이 더 잘 맞는가?", fontsize=14)
ax2.legend(fontsize=11)
ax2.grid(alpha=0.3)

# Compute RMS errors.
rms_rad = np.sqrt(np.mean(resid_rad**2))
rms_adi = np.sqrt(np.mean(resid_adi**2))
ax2.text(0.5, 0.3, f'RMS (Radiative) = {rms_rad:.3f}\n'
                     f'RMS (Adiabatic) = {rms_adi:.3f}',
         fontsize=12, fontweight='bold',
         bbox=dict(boxstyle='round', fc='lightyellow'))

plt.tight_layout()
plt.show()

print(f"\nRMS 오차 비교:")
print(f"  복사 평형: {rms_rad:.4f}")
print(f"  단열 평형: {rms_adi:.4f}")
print(f"  → 복사 평형이 {rms_adi/rms_rad:.1f}배 더 잘 맞음!")
print(f"\nSchwarzschild의 결론: 태양 외층은 복사 평형에 가깝다.")

## Part 6: Stability Analysis / 안정성 분석

복사 평형이 안정하려면, 복사 온도 경사가 단열 온도 경사보다 작아야 합니다.

For radiative equilibrium to be stable, the radiative temperature gradient must be less steep than the adiabatic gradient.

안정성 조건 / Stability condition (eq. 17): $\quad 1 - \dfrac{\tau^4}{t^4} < 4\dfrac{k-1}{k}$

$k > 4/3$이면 항상 만족. $k \leq 4/3$이면 깊은 곳에서 불안정 가능.

Always satisfied for $k > 4/3$. Possible instability at depth for $k \leq 4/3$.

In [ ]:
# Stability analysis: compare radiative and adiabatic temperature gradients.

t_range = np.linspace(TAU_BOUNDARY, 30000, 500)  # Temperature range.
tau = TAU_BOUNDARY

# Left side of stability condition: 1 - tau^4/t^4.
lhs = 1 - tau**4 / t_range**4

# Right side for different gas types.
k_values = {'Monatomic (k=5/3)\n단원자': 5/3,
            'Diatomic (k=7/5)\n이원자': 7/5,
            'Triatomic (k=4/3)\n삼원자': 4/3,
            'Polyatomic (k≈1.2)\n다원자': 1.2}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Left: Stability condition as function of temperature.
ax1.plot(t_range, lhs, 'k-', linewidth=2.5,
         label=r'$1 - \tau^4/t^4$ (radiative gradient / 복사 경사)')

colors = ['blue', 'green', 'red', 'orange']
for (name, k), color in zip(k_values.items(), colors):
    rhs = 4 * (k - 1) / k
    ax1.axhline(rhs, color=color, linestyle='--', linewidth=1.5,
                label=f'{name}: 4(k-1)/k = {rhs:.2f}')

ax1.axhline(1.0, color='gray', linestyle=':', alpha=0.5)
ax1.text(28000, 1.02, 'asymptote = 1', fontsize=9, color='gray')
ax1.set_xlabel("Temperature t / 온도 (K)", fontsize=13)
ax1.set_ylabel("Gradient ratio / 경사 비율", fontsize=13)
ax1.set_title("Stability Condition (eq. 17)\n"
              "안정성 조건 — 선 아래 = 안정, 위 = 불안정",
              fontsize=14)
ax1.legend(fontsize=9, loc='lower right')
ax1.grid(alpha=0.3)
ax1.set_ylim(0, 2.5)

# Shade stable/unstable regions for diatomic gas.
k_di = 7 / 5
rhs_di = 4 * (k_di - 1) / k_di
# Find crossing point.
t_cross = tau / (1 - rhs_di) ** 0.25 if rhs_di < 1 else None
if t_cross and t_cross < 30000:
    ax1.axvline(t_cross, color='green', alpha=0.3, linestyle='-.')
    ax1.fill_between(t_range[t_range > t_cross], lhs[t_range > t_cross],
                     rhs_di, where=lhs[t_range > t_cross] > rhs_di,
                     alpha=0.1, color='red')
    ax1.text(t_cross + 500, 0.5, f'Unstable for\ndiatomic at\nt > {t_cross:.0f} K',
             fontsize=9, color='green')

# Right: Solar interior schematic.
ax2.set_aspect('equal')
theta = np.linspace(0, 2 * np.pi, 100)

# Draw Sun layers.
for r, color, label in [(1.0, '#FFD700', 'Photosphere / 광구'),
                         (0.7, '#FFA500', 'Convection zone / 대류층'),
                         (0.25, '#FF4500', 'Radiative zone / 복사층'),
                         (0.1, '#FF0000', 'Core / 핵')]:
    circle = plt.Circle((0, 0), r, fill=True, color=color, alpha=0.4)
    ax2.add_patch(circle)

ax2.text(0, 0.85, 'Convection Zone\n대류층 (0.7-1.0 R☉)\nAdiabatic eq.',
         ha='center', fontsize=9, fontweight='bold')
ax2.text(0, 0.45, 'Radiative Zone\n복사층 (0.25-0.7 R☉)\nRadiative eq.',
         ha='center', fontsize=9, fontweight='bold')
ax2.text(0, 0.0, 'Core\n핵', ha='center', fontsize=9, fontweight='bold')

# Arrow showing Schwarzschild's prediction.
ax2.annotate('', xy=(0.7, -0.5), xytext=(1.0, -0.5),
             arrowprops=dict(arrowstyle='<->', color='darkred', lw=2))
ax2.text(0.85, -0.6, 'Schwarzschild\npredicted this\nboundary!\n이 경계를\n예견!',
         ha='center', fontsize=8, color='darkred', fontweight='bold')

ax2.set_xlim(-1.2, 1.2)
ax2.set_ylim(-0.8, 1.2)
ax2.set_title("Modern Solar Structure\n현대 태양 구조 — Schwarzschild가 예견",
              fontsize=14)
ax2.axis('off')

plt.tight_layout()
plt.show()

print("Schwarzschild의 안정성 분석 결론:")
print("  1. k > 4/3 (삼원자 이상): 복사 평형이 모든 깊이에서 안정")
print("  2. k = 7/5 (이원자): 깊은 곳에서 대류 불안정 발생 가능")
print("  → '바깥 껍질은 복사 평형, 내부는 대류 영역' 예측")
print("  → 현대에 확인된 복사층/대류층 경계(0.7 R☉)의 최초 이론적 예견!")

## Part 7: Solar Disk Visualization / 태양 원반 시각화

두 모델의 주변감광을 태양 원반 이미지로 시각화합니다. 중심이 밝고 가장자리가 어두운 효과를 직접 눈으로 확인할 수 있습니다.

Visualize limb darkening as solar disk images for both models. The center-bright, limb-dark effect is directly visible.

In [ ]:
# Create synthetic solar disk images with limb darkening.

def make_solar_disk(n_pixels=500, darkening_func=None):
    """Generate a 2D solar disk image with limb darkening.

    Args:
        n_pixels: Image size in pixels.
        darkening_func: Function mapping r/R to brightness (0-1).

    Returns:
        2D numpy array representing the solar disk.
    """
    y, x = np.mgrid[-1:1:complex(n_pixels), -1:1:complex(n_pixels)]
    r = np.sqrt(x**2 + y**2)
    disk = np.zeros((n_pixels, n_pixels))
    inside = r < 1.0
    if darkening_func is not None:
        disk[inside] = darkening_func(r[inside])
    else:
        disk[inside] = 1.0
    return disk


fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# No limb darkening.
disk_uniform = make_solar_disk(darkening_func=lambda r: np.ones_like(r))
axes[0].imshow(disk_uniform, cmap='hot', vmin=0, vmax=1, extent=[-1, 1, -1, 1])
axes[0].set_title("No Limb Darkening\n주변감광 없음 (균일)", fontsize=13)

# Radiative equilibrium.
disk_rad = make_solar_disk(darkening_func=limb_darkening_radiative)
axes[1].imshow(disk_rad, cmap='hot', vmin=0, vmax=1, extent=[-1, 1, -1, 1])
axes[1].set_title("Radiative Equilibrium (eq. 28)\n복사 평형: F = (1+2cosθ)/3",
                   fontsize=13)

# Adiabatic equilibrium.
disk_adi = make_solar_disk(darkening_func=limb_darkening_adiabatic)
axes[2].imshow(disk_adi, cmap='hot', vmin=0, vmax=1, extent=[-1, 1, -1, 1])
axes[2].set_title("Adiabatic Equilibrium (eq. 29)\n단열 평형: F = cosθ",
                   fontsize=13)

for ax in axes:
    ax.set_xlabel("x / R☉")
    ax.set_ylabel("y / R☉")

fig.suptitle("Solar Disk Appearance Under Different Models (Schwarzschild 1906)\n"
             "다른 모델에서의 태양 원반 모습",
             fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("시각적 비교:")
print("  균일: 가장자리와 중심이 같은 밝기 (비현실적)")
print("  복사 평형: 가장자리가 약간 어두움 (1/3) — 실제 태양과 유사!")
print("  단열 평형: 가장자리가 완전히 어두움 (0) — 너무 극단적")

## Part 8: Summary / 요약

| Part | 구현 내용 / Implementation | 핵심 원리 / Core Principle | 다음 논문 연결 / Next Paper |
|------|---------|---------|---------------|
| 1 | 세 가지 평형의 온도 프로파일 / Temperature profiles | 등온 vs 단열 vs 복사 / Three equilibria | #5 Hale: 태양 대기 구조가 흑점 스펙트럼 해석의 맥락 |
| 2 | 복사 전달 방정식 수치 해 / RT equation solver | $dB/dm = E-B$, $A+B=2E$ | Chandrasekhar: 산란 포함 완전 이론 |
| 3–5 | 주변감광 예측 & 관측 비교 / Limb darkening | $F(i) = (1+2\cos i)/3$ | 현대 SDO 관측과의 비교 |
| 6 | 안정성 분석 / Stability | $1-\tau^4/t^4 < 4(k-1)/k$ | #17 Schou: 타코클라인 발견으로 예측 확인 |
| 7 | 태양 원반 2D 시각화 / Solar disk visualization | 주변감광의 시각적 효과 / Visual effect | 실제 태양 관측 영상과의 비교 |

**Schwarzschild (1906)의 유산 / Legacy**: 13쪽의 짧은 논문으로 복사 전달 이론을 탄생시키고, 태양 대기의 온도 구조를 최초로 정량적으로 모델링하며, 대류/복사 영역 구분을 예견한 — 천체물리학의 기념비적 업적.

A 13-page paper that birthed radiative transfer theory, first quantitatively modeled the solar atmospheric temperature structure, and predicted the convection/radiation zone boundary — a monumental achievement in astrophysics.